In [ ]:
# # Not using right now, but this is how to stream the dataset and display it in a DataFrame
# from datasets import load_dataset
# import pandas as pd

# # Stream the dataset
# ds = load_dataset("OpenPipe/hacker-news", split="train", streaming=True)

# # Number of posts you want
# N = 10
# sample_posts = []

# for post in ds:
#     # Only keep posts with non-empty text
#     text = post.get("text")
#     if text and text.strip():
#         sample_posts.append(post)
#     if len(sample_posts) >= N:
#         break

# # Convert to DataFrame
# df = pd.DataFrame(sample_posts)

# # Keep only columns that exist
# #desired_cols = ["title", "url", "text", "time"]
# #cols_to_keep = [c for c in desired_cols if c in df.columns]
# #df = df[cols_to_keep]

# # Optional: shorten text for display
# df["text"] = df["text"].str.slice(0, 300)

# # Convert timestamp to string if it's not already
# if "time" in df.columns:
#     df["time"] = df["time"].apply(
#         lambda t: t.strftime('%Y-%m-%d %H:%M:%S') if hasattr(t, "strftime") else str(t)
#     )

# # Display nicely
# df

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/39 [00:00<?, ?it/s]

,id,type,by,time,title,text,url,score,parent,top_level_parent,descendants,kids,deleted,dead
0,15,comment,sama,2006-10-09 19:51:01,None,&#34;the rising star of venture capital&#34; -...,None,None,1,1,None,[17],None,None
1,17,comment,pg,2006-10-09 19:52:45,None,Is there anywhere to eat on Sandhill Road?,None,None,15,1,None,[1079],None,None
2,22,comment,pg,2006-10-10 02:18:22,None,It's kind of funny that Sevin Rosen is giving ...,None,None,21,21,None,None,None,None
3,23,comment,starklysnarky,2006-10-10 02:30:53,None,"This is interesting, but the limitations becom...",None,None,20,20,None,None,None,None
4,30,comment,spez,2006-10-10 15:34:59,None,Stay tuned...,None,None,29,29,None,[31],None,None
5,31,comment,pg,2006-10-10 15:40:05,None,I'm tuned...,None,None,30,29,None,[33],None,None
6,33,comment,spez,2006-10-10 15:50:40,None,winnar winnar chicken dinnar!,None,None,31,29,None,[34],None,None
7,34,comment,pg,2006-10-10 15:53:53,None,what do you mean? this story's still not #1,None,None,33,29,None,"[36, 35, 531706]",None,None
8,35,comment,spez,2006-10-10 15:57:42,None,perhaps if i hadn't told you it was coming\r\n...,None,None,34,29,None,None,None,None
9,36,comment,pg,2006-10-10 16:01:01,None,Can you do it again?,None,None,34,29,None,None,None,None


In [2]:
from datasets import load_dataset
import pandas as pd

In [2]:
# Import media reliability dataset in full

media_ds = load_dataset("sergioburdisso/news_media_reliability", split="train")

media_df = pd.DataFrame(media_ds)

media_df

media_lookup = {
    row['domain']: {
        "reliability_label": row['reliability_label'],
        "newsguard_score": row['newsguard_score']
    }
    for _, row in media_df.iterrows()
}

In [3]:
# Now we can use media_lookup to add reliability info to our news articles when we stream them
# This is a more complex example that also extracts dates from the headline and text, and adds the news site domain. We can then merge this with the media reliability info based on the domain.
# Note: this is just an example of how to do it, and may need adjustments based on the actual structure of the news articles in the dataset.
from datasets import load_dataset
import pandas as pd
import re
from dateutil import parser
from urllib.parse import urlparse
import random

# Stream the dataset
ds = load_dataset("ashraq/financial-news-articles", split="train", streaming=True)

# Company we want to look at
company = "all"   # change to "all" to pull everything

# Options
sample = True       # whether to sample the matched articles
sample_frac = 0.1   # fraction to keep if sampling
# N = 10            # optional: max number of collected articles

sample_articles = []

def extract_domain(url):
    if not url:
        return None
    return urlparse(url).netloc.replace("www.", "")

def extract_date(text):
    if not text:
        return None

    date_patterns = [
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}',
        r'\b\d{4}-\d{2}-\d{2}',
        r'\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}'
    ]

    for pattern in date_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            try:
                return parser.parse(match.group(), fuzzy=True)
            except:
                continue

    return None

company_lower = company.lower()

for article in ds:
    headline = article.get("title", "")
    text = article.get("text", "")

    matches = (
        company_lower == "all"
        or company_lower in headline.lower()
        or company_lower in text.lower()
    )

    if matches:
        # Add news_site
        article["news_site"] = extract_domain(article.get("url"))

        # Add extracted dates
        article["title_date"] = extract_date(headline)
        article["text_date"] = extract_date(text)
        
        domain = article["news_site"]
        if domain in media_lookup:
            article["reliability_label"] = media_lookup[domain]["reliability_label"]
            article["newsguard_score"] = media_lookup[domain]["newsguard_score"]
        else:
            article["reliability_label"] = None
            article["newsguard_score"] = None

        sample_articles.append(article)

# Optional sampling after streaming
if sample and sample_articles:
    k = max(1, int(len(sample_articles) * sample_frac))
    sample_articles = random.sample(sample_articles, k)

# Convert to DataFrame (optional, just for viewing)
df = pd.DataFrame(sample_articles)

df

,title,text,url,news_site,title_date,text_date,reliability_label,newsguard_score
0,BRIEF-David Tepper Agrees To Buy Carolina Pant...,May 15 (Reuters) -\n* HEDGE FUND OWNER DAVID T...,https://www.reuters.com/article/brief-david-te...,reuters.com,None,None,1.0,100.0
1,Adverum Biotechnologies Announces Leadership C...,May 3 (Reuters) - Adverum Biotechnologies Inc:...,https://www.reuters.com/article/brief-adverum-...,reuters.com,None,None,1.0,100.0
2,BRIEF-Palaces Real Estate and Development FY L...,Jan 30 (Reuters) - PALACES REAL ESTATE AND DEV...,https://www.reuters.com/article/brief-palaces-...,reuters.com,None,None,1.0,100.0
3,Julia Louis-Dreyfus to receive Mark Twain Priz...,"May 23, 2018 / 5:38 PM / Updated 25 minutes ag...",https://uk.reuters.com/article/uk-people-julia...,uk.reuters.com,None,2018-05-23 00:00:00,NaN,NaN
4,Russian bonds join key Bloomberg-Barclays inde...,"LONDON, Feb 28 (Reuters) - Russia’s foreign cu...",https://www.reuters.com/article/russia-bonds/r...,reuters.com,None,None,1.0,100.0
...,...,...,...,...,...,...,...,...
30619,"Energizer Holdings, Inc. Announces Agreement T...","ST. LOUIS, Jan. 16, 2018 /PRNewswire/ -- Energ...",http://www.cnbc.com/2018/01/16/pr-newswire-ene...,cnbc.com,None,2018-01-16 00:00:00,1.0,95.0
30620,BRIEF-HM Inwest To Issue Series C Bonds Of Tot...,Jan 30 (Reuters) - HM INWEST SA:\n* DECIDES TO...,https://www.reuters.com/article/brief-hm-inwes...,reuters.com,None,None,1.0,100.0
30621,Watts Water Technologies Announces Departure o...,"NORTH ANDOVER, Mass.--(BUSINESS WIRE)-- Watts ...",http://www.cnbc.com/2018/03/30/business-wire-w...,cnbc.com,None,2018-04-06 00:00:00,1.0,95.0
30622,MasTec Completes Existing Common Stock Repurch...,"CORAL GABLES, Fla., March 29, 2018 /PRNewswire...",http://www.cnbc.com/2018/03/29/pr-newswire-mas...,cnbc.com,None,2018-03-29 00:00:00,1.0,95.0


In [10]:
# Now we can do some basic sentiment/direction classification based on the presence of certain keywords in the headline and text. 
# This is a very simple approach and can be improved with more sophisticated NLP techniques, but it gives us a starting point for analysis.

NEG_CUES = r"\b(decrease|decline|down|lower|pressure|headwind|soft|weak|challenging|unfavorable|adverse|constraint|shortage)\b"
POS_CUES = r"\b(increase|grow|up|higher|strong|improve|favorable|benefit|tailwind|expand|expansion|accelerate|record)\b"

def classify_direction(s: str) -> str:
    s_l = s.lower()
    neg = len(re.findall(NEG_CUES, s_l))
    pos = len(re.findall(POS_CUES, s_l))
    if neg > pos and neg >= 1:
        return "negative"
    if pos > neg and pos >= 1:
        return "positive"
    return "neutral"
df["headline_direction"] = df["title"].apply(classify_direction)
df["text_direction"] = df["text"].apply(classify_direction)
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30624 entries, 0 to 30623
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               30624 non-null  object 
 1   text                30624 non-null  object 
 2   url                 30624 non-null  object 
 3   news_site           30624 non-null  object 
 4   title_date          406 non-null    object 
 5   text_date           13884 non-null  object 
 6   reliability_label   25495 non-null  float64
 7   newsguard_score     25495 non-null  float64
 8   headline_direction  30624 non-null  object 
 9   text_direction      30624 non-null  object 
dtypes: float64(2), object(8)
memory usage: 2.3+ MB


In [11]:
# Adjust the schema to match Edgar data
df['cik'] = None
df['accession_number'] = None
df['section'] = 'news_article'
df['form_type'] = 'news_article'
df.head()

,title,text,url,news_site,title_date,text_date,reliability_label,newsguard_score,headline_direction,text_direction,cik,accession_number,section,form_type
0,BRIEF-David Tepper Agrees To Buy Carolina Pant...,May 15 (Reuters) -\n* HEDGE FUND OWNER DAVID T...,https://www.reuters.com/article/brief-david-te...,reuters.com,None,None,1.0,100.0,positive,positive,None,None,news_article,news_article
1,Adverum Biotechnologies Announces Leadership C...,May 3 (Reuters) - Adverum Biotechnologies Inc:...,https://www.reuters.com/article/brief-adverum-...,reuters.com,None,None,1.0,100.0,neutral,negative,None,None,news_article,news_article
2,BRIEF-Palaces Real Estate and Development FY L...,Jan 30 (Reuters) - PALACES REAL ESTATE AND DEV...,https://www.reuters.com/article/brief-palaces-...,reuters.com,None,None,1.0,100.0,neutral,neutral,None,None,news_article,news_article
3,Julia Louis-Dreyfus to receive Mark Twain Priz...,"May 23, 2018 / 5:38 PM / Updated 25 minutes ag...",https://uk.reuters.com/article/uk-people-julia...,uk.reuters.com,None,2018-05-23 00:00:00,NaN,NaN,neutral,positive,None,None,news_article,news_article
4,Russian bonds join key Bloomberg-Barclays inde...,"LONDON, Feb 28 (Reuters) - Russia’s foreign cu...",https://www.reuters.com/article/russia-bonds/r...,reuters.com,None,None,1.0,100.0,neutral,neutral,None,None,news_article,news_article


In [ ]:
# Now we can try to match the company names in the articles to the SEC company tickers and CIKs. 
# This is a very basic approach that just looks for exact matches of the company name in the headline and text, 
# but it can be improved with more sophisticated entity recognition techniques.

import json
import gzip
import urllib.request

url = "https://www.sec.gov/files/company_tickers.json"
headers = {
    "User-Agent": "Chase chasecapanna@berkeley.edu",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov",
}

req = urllib.request.Request(url, headers=headers)

with urllib.request.urlopen(req, timeout=30) as r:
    data = r.read()
    if r.headers.get("Content-Encoding", "").lower() == "gzip":
        data = gzip.decompress(data)
    raw = json.loads(data.decode("utf-8"))

sec_companies = (
    pd.DataFrame.from_dict(raw, orient="index")
    .rename(columns={"title": "company_name", "cik_str": "cik"})
)

sec_companies["ticker"] = sec_companies["ticker"].astype(str).str.upper().str.strip()
sec_companies["company_name"] = sec_companies["company_name"].astype(str).str.strip()

sec_companies[["ticker", "company_name", "cik"]].head()


,ticker,company_name,cik
0,NVDA,NVIDIA CORP,1045810
1,AAPL,Apple Inc.,320193
2,GOOGL,Alphabet Inc.,1652044
3,MSFT,MICROSOFT CORP,789019
4,AMZN,AMAZON COM INC,1018724


In [ ]:
# We can build a lookup index that maps normalized company name variants to their corresponding tickers and CIKs.
# This allows us to extract potential company matches from the article headlines and text, and link them to the SEC data for further analysis.
import re
from collections import defaultdict

# ---------- helpers ----------
suffix_re = re.compile(
    r"\b(inc|incorporated|corp|corporation|co|company|ltd|limited|plc|ag|nv|sa|group|holdings|holding)\.?$",
    flags=re.IGNORECASE,
)

def norm(s: str) -> str:
    s = re.sub(r"[^a-z0-9 ]+", " ", str(s).lower())
    return re.sub(r"\s+", " ", s).strip()

def aliases(company_name: str):
    base = norm(company_name)
    if not base:
        return set()
    out = {base}
    while True:
        stripped = suffix_re.sub("", base).strip()
        if stripped == base or not stripped:
            break
        out.add(stripped)
        base = stripped
    return {x for x in out if len(x) >= 3}

# ---------- build lookup index from SEC mapping ----------
# sec_companies must contain: company_name, ticker, cik
alias_to_rows = defaultdict(list)
first_token_index = defaultdict(list)

for _, r in sec_companies[["company_name", "ticker", "cik"]].dropna().iterrows():
    for a in aliases(r["company_name"]):
        alias_to_rows[a].append((r["company_name"], r["ticker"], r["cik"]))

for a, rows in alias_to_rows.items():
    first = a.split(" ", 1)[0]
    first_token_index[first].append((a, rows))

# ---------- extract matches from title + text ----------
def extract_company_matches(title, text):
    combined = norm(f"{title or ''} {text or ''}")
    if not combined:
        return []
    padded = f" {combined} "
    tokens = set(combined.split())

    matches = []
    seen = set()
    for t in tokens:
        for a, rows in first_token_index.get(t, []):
            if f" {a} " in padded:
                for company_name, ticker, cik in rows:
                    key = (company_name, ticker, cik)
                    if key not in seen:
                        seen.add(key)
                        matches.append(key)
    return matches

df["company_matches"] = df.apply(
    lambda r: extract_company_matches(r.get("title", ""), r.get("text", "")),
    axis=1
)

In [ ]:
# Now we can explode the matches into separate rows so we can analyze them more easily.
# Note: this will create multiple rows for articles that mention multiple companies, which is useful for linking to the SEC data.

df_long = df.explode("company_matches").copy()
df_long[["company_name", "ticker", "cik"]] = pd.DataFrame(
    df_long["company_matches"].tolist(), index=df_long.index
)
df_long = df_long.drop(columns=["company_matches"])


In [22]:
# Reorder columns to put Edgar-related ones first
edgar_order = [
    "cik",
    "ticker",
    "company_name",
    "sic",
    "sic_description",
    "entity_type",
    "industry_group",
    "manual_include",
]

front = [c for c in edgar_order if c in df_long.columns]
rest = [c for c in df_long.columns if c not in front]

df_long = df_long[front + rest]


In [65]:
df_long.to_parquet("ashraq_fin_news.parquet", engine="pyarrow", index=False)

# EVERYTHING PAST THIS IS NOT WORKING / TAKES TOO LONG

In [24]:
from datasets import load_dataset

ds = load_dataset("cc_news", split="train")
print(ds[0])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00005.parquet:   0%|          | 0.00/211M [00:00<?, ?B/s]

plain_text/train-00001-of-00005.parquet:   0%|          | 0.00/234M [00:00<?, ?B/s]

plain_text/train-00002-of-00005.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

plain_text/train-00003-of-00005.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

plain_text/train-00004-of-00005.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/708241 [00:00<?, ? examples/s]

{'title': 'Daughter Duo is Dancing in The Same Company', 'text': 'There\'s a surprising twist to Regina Willoughby\'s last season with Columbia City Ballet: It\'s also her 18-year-old daughter Melina\'s first season with the company. Regina, 40, will retire from the stage in March, just as her daughter starts her own career as a trainee. But for this one season, they\'re sharing the stage together.\nPerforming Side-By-Side In The Nutcracker\nRegina and Melina are not only dancing in the same Nutcracker this month, they\'re onstage at the same time: Regina is doing Snow Queen, while Melina is in the snow corps, and they\'re both in the Arabian divertissement. "It\'s very surreal to be dancing it together," says Regina. "I don\'t know that I ever thought Melina would take ballet this far."\nLeft: Regina and Melina with another company member post-snow scene in 2003. Right: The pair post-snow scene in 2017 (in the same theater)\nKeep reading at dancemagazine.com.', 'domain': 'www.pointema

In [ ]:
finance = ds.filter(lambda x: "earnings" in x["text"].lower())


Filter:   0%|          | 0/708241 [00:00<?, ? examples/s]

In [29]:
df = finance.to_pandas()
df.sort_values("date", inplace=True, ascending=False)
df.head()

,title,text,domain,date,description,url,image_url
3582,Volvo Car Mobility launches mobility brand M,"Volvo Cars, the premium car maker, today launc...",www.automotiveworld.com,2018-07-11 00:00:00,"Volvo Cars, the premium car maker, today launc...",https://www.automotiveworld.com/news-releases/...,https://1bgnvt3q09toyb96v2ecsygm-wpengine.netd...
7206,Outcry as Japanese winemaking couple ordered t...,"From their vineyards in southern France, the S...",japantoday.com,2018-07-05 06:43:00,French wine lovers have risen up in revolt aft...,https://japantoday.com/category/business/Outcr...,https://japantoday-asset.scdn3.secure.raxcdn.c...
3569,HDFC seeks opportunities in stressed real estate,"Deepak Parekh, chairman, HDFC\nHousing Develop...",www.financialexpress.com,2018-07-05 02:19:01,Housing Development Finance Corporation (HDFC)...,https://www.financialexpress.com/industry/hdfc...,https://images.financialexpress.com/2018/07/de...
3396,India should leverage its livestock treasure b...,Indian buffaloes are currently exported to abo...,indianexpress.com,2018-07-05 02:11:58,Dairying cannot sustain itself without a vibra...,https://indianexpress.com/article/india/india-...,https://images.indianexpress.com/2018/07/dairy...
3274,Content-hungry bidders circle 'Big Brother' ma...,"Several bidders, including Liberty Global, are...",www.channelnewsasia.com,2018-07-05 01:35:48,"Several bidders, including Liberty Global, are...",https://www.channelnewsasia.com/news/business/...,http://www.channelnewsasia.com/image/10500142/...


In [40]:
from newspaper import Article
import feedparser
import pandas as pd
from datetime import datetime, timedelta

# RSS feeds to scrape
rss_feeds = [
    {"url": "https://www.cnbc.com/id/100003114/device/rss/rss.html", "source": "CNBC"},
    {"url": "https://seekingalpha.com/feed.xml", "source": "Seeking Alpha"},
    {"url": "https://feeds.marketwatch.com/marketwatch/topstories/", "source": "MarketWatch"}
]

# How many days back to fetch
days_back = 365
end_date = datetime.now()
start_date = end_date - timedelta(days=days_back)

articles = []

for feed_info in rss_feeds:
    feed = feedparser.parse(feed_info["url"])
    source_name = feed_info["source"]

    for entry in feed.entries:
        published = getattr(entry, "published", None)
        if not published:
            continue

        # Convert RSS published time to datetime
        entry_date = datetime(*entry.published_parsed[:6])

        # Skip if not within the last N days
        if not (start_date <= entry_date <= end_date):
            continue

        url = entry.link
        article = Article(url)
        try:
            article.download()
            article.parse()
        except:
            continue  # skip if scraping fails

        if article.text:
            articles.append({
                "title": entry.title,
                "date": entry_date,
                "url": url,
                "text": article.text,
                "source": source_name
            })

# Create DataFrame
df = pd.DataFrame(articles)
df = df.reset_index(drop=True)

print(f"Total articles collected: {len(df)}")

Total articles collected: 16


In [41]:
df.head()

,title,date,url,text,source
0,Five races to watch on the day the 2026 midter...,2026-03-03 17:31:21,https://www.cnbc.com/2026/03/03/2026-midterm-e...,watch now\n\nTuesday is the first moment of tr...,CNBC
1,Blackstone’s Gray: Market ‘noise’ fueled recor...,2026-03-03 18:55:30,https://www.cnbc.com/2026/03/03/blackstone-pri...,"Jon Gray, President and COO of Blackstone, spe...",CNBC
2,Apple raises MacBook prices across the board a...,2026-03-03 18:49:45,https://www.cnbc.com/2026/03/03/apple-macbook-...,In this article AAPL Follow your favorite stoc...,CNBC
3,"Banking, payments services disrupted after Ama...",2026-03-03 16:51:37,https://www.cnbc.com/2026/03/03/iran-war-uae-d...,Foreign workers look at a tall plume of black ...,CNBC
4,Anthropic 'made a mistake' in Pentagon talks a...,2026-03-03 16:21:48,https://www.cnbc.com/2026/03/03/anthropic-pent...,FCC Chairman Brendan Carr testifies before the...,CNBC


In [47]:
API_KEY = "a6a36f86d7c741688baec7111495a9f0"

Domains:   0%|          | 0/3 [00:00<?, ?it/s]

Error fetching cnbc.com page 1: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching cnbc.com page 2: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching cnbc.com page 3: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching cnbc.com page 4: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request res

Domains:  33%|███▎      | 1/3 [00:00<00:01,  1.18it/s]

Error fetching cnbc.com page 5: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching marketwatch.com page 1: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching marketwatch.com page 2: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching marketwatch.com page 3: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are 

Domains:  67%|██████▋   | 2/3 [00:01<00:00,  1.05it/s]

Error fetching marketwatch.com page 4: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching marketwatch.com page 5: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching seekingalpha.com page 1: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Error fetching seekingalpha.com page 2: {'status': 'error', 'code': 'parameterInvalid', 'message': 

Domains: 100%|██████████| 3/3 [00:02<00:00,  1.07it/s]

Error fetching seekingalpha.com page 5: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2026-02-03, but you have requested 2020-01-01. You may need to upgrade to a paid plan.'}
Total articles collected: 0


""


In [59]:
domains = ["cnbc.com", "marketwatch.com", "seekingalpha.com"]

# CC-MAIN crawl batches (monthly-ish) 2020–2026
# Replace with actual batch names as you find them in https://commoncrawl.org/the-data/get-started/
batches = [
    "CC-MAIN-2020-05", "CC-MAIN-2020-13", "CC-MAIN-2021-04",
    "CC-MAIN-2021-22", "CC-MAIN-2022-05", "CC-MAIN-2022-16",
    "CC-MAIN-2023-05", "CC-MAIN-2023-13", "CC-MAIN-2025-13"
]

In [60]:
from cdx_toolkit import CDXFetcher
from tqdm import tqdm

fetcher = CDXFetcher(source='cc')
all_records = []

for batch in batches:
    for domain in domains:
        print(f"Fetching {domain} from {batch}...")
        try:
            records = fetcher.iter(
                url=f"{domain}/*",
                collection=batch
            )
            all_records.extend(list(records))
        except Exception as e:
            print(f"Error fetching {domain} from {batch}: {e}")

print(f"Total matching records found: {len(all_records)}")

Fetching cnbc.com from CC-MAIN-2020-05...


KeyboardInterrupt: 

In [ ]:
from newspaper import Article
import pandas as pd
from tqdm import tqdm

articles = []

for rec in tqdm(all_records, desc="Downloading and parsing HTML"):
    try:
        resp = rec.fetch()
        html = resp.content.decode('utf-8', errors='ignore')
        
        a = Article(rec.url)
        a.set_html(html)
        a.parse()
        
        if len(a.text) > 200:  # skip very short pages
            articles.append({
                "url": rec.url,
                "date": rec.timestamp,
                "text": a.text,
                "source": rec.url.split("/")[2]
            })
    except Exception:
        continue  # skip problematic pages

df = pd.DataFrame(articles)
df = df.drop_duplicates(subset="url").reset_index(drop=True)

df.to_parquet("financial_articles_2020_2026.parquet")
print(f"Total articles saved: {len(df)}")

In [64]:
from cdx_toolkit import CDXFetcher

fetcher = CDXFetcher(source='cc')
domain = "cnbc.com"
collection = "CC-MAIN-2025-13"

records = []
for i, rec in enumerate(fetcher.iter(url="cnbc.com/*", collection="CC-MAIN-2025-13")):
    if i >= 20:
        break
    records.append(rec.url)
    print(rec.url, rec.timestamp)
print("Sample URLs:", records)

KeyboardInterrupt: 

In [ ]:
from newspaper import Article

sample_urls = records[:5]  # 5 pages for super-fast test
articles = []

for url in sample_urls:
    try:
        a = Article(url)
        a.download()
        a.parse()
        articles.append({
            "url": url,
            "text": a.text,
            "source": url.split("/")[2]
        })
    except:
        continue

print(len(articles))

In [5]:
df_cnbc = pd.read_parquet("parquet_by_domain/cnbc_com_cleaned.parquet", engine="pyarrow")

df_cnbc.info()
df_cnbc.head()

<class 'pandas.DataFrame'>
RangeIndex: 236722 entries, 0 to 236721
Data columns (total 27 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   cik                 231333 non-null  float64       
 1   ticker              231333 non-null  str           
 2   accession_number    0 non-null       object        
 3   form_type           236722 non-null  str           
 4   filing_date         25174 non-null   datetime64[us]
 5   section             236722 non-null  str           
 6   fact_type           236722 non-null  str           
 7   direction           236722 non-null  str           
 8   evidence_text       236722 non-null  str           
 9   source_url          236722 non-null  str           
 10  sent_index          0 non-null       object        
 11  url                 236722 non-null  str           
 12  url_mobile          222061 non-null  str           
 13  title               236722 non-null  str

,cik,ticker,accession_number,form_type,filing_date,section,fact_type,direction,evidence_text,source_url,...,language,sourcecountry,text,source_file,news_site,title_date,text_date,headline_direction,text_direction,company_name
0,1026785.0,HIHO,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,HIGHWAY HOLDINGS LTD
1,320193.0,AAPL,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,Apple Inc.
2,1318605.0,TSLA,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,"Tesla, Inc."
3,1823652.0,EVEX,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,"Eve Holding, Inc."
4,1823652.0,EVEX-WT,None,news_article,2018-03-23,news_article,news_article,neutral,The U.S. National Highway Traffic Safety Admin...,https://www.cnbc.com/2020/01/01/nhtsa-will-pro...,...,English,United States,The U.S. National Highway Traffic Safety Admin...,2020-01-01.csv,cnbc.com,NaT,2018-03-23,neutral,neutral,"Eve Holding, Inc."
